## csvをparquetに変換

In [ ]:
import pandas as pd


# CSVファイルを読み込む
df = pd.read_csv('../dataset/train.csv')

# Parquetファイルとして保存
df.to_parquet('../dataset/train.parquet', index=False)

/tmp/ipykernel_13002/3958761153.py:5: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dataset/train.csv')


KeyboardInterrupt: 

In [ ]:
import pandas as pd


# CSVファイルを読み込む
df = pd.read_csv('../dataset/test.csv')

df["unit_name"] = df["unit_name"].astype(str)
# Parquetファイルとして保存
df.to_parquet('../dataset/test.parquet', index=False)

/tmp/ipykernel_2841/338712073.py:5: DtypeWarning: Columns (46,55,56,63,146) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dataset/test.csv')


In [ ]:
df.shape

(112437, 149)

## 前処理

In [1]:
# 必要なライブラリの読み込み
import gc
import numpy as np
import pandas as pd
import warnings
import geopandas as gpd
from shapely.geometry import Point

warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
# データディレクトリの設定
ROOT_DIR = "../dataset"

In [ ]:
# データの読み込み
train_df = pd.read_parquet(f"{ROOT_DIR}/train.parquet")
test_df = pd.read_parquet(f"{ROOT_DIR}/test.parquet")

In [ ]:
train_df.shape,  test_df.shape

((363924, 149), (112437, 149))

In [53]:
# int型_NaNを埋めてから整数型に変換（0や99など変数にあわせて設定）
test_df['floor_count'] = test_df['floor_count'].fillna(0).astype(int)
test_df['walk_distance1'] = test_df['walk_distance1'].fillna(9999).astype(int)
test_df['walk_distance2'] = test_df['walk_distance2'].fillna(9999).astype(int)
test_df["madori_kind_all"] = test_df["madori_kind_all"].fillna(0).astype(int)
test_df["madori_number_all"] = test_df["madori_number_all"].fillna(0).astype(int)

# float型をint型に変換
test_df["building_structure"] = test_df["building_structure"].fillna(0).astype(int)
test_df["money_kyoueki"] = test_df["money_kyoueki"].fillna(0).astype(int)
test_df["money_kyoueki_tax"] = test_df["money_kyoueki_tax"].fillna(0).astype(int)
test_df["post1"] = test_df["post1"].fillna(0).astype(int)
test_df["room_kaisuu"] = test_df["room_kaisuu"].fillna(0).astype(int)

# object型のnanを"NULL"に変換
test_df['building_tag_id'] = test_df['building_tag_id'].fillna("NA")
test_df['eki_name1'] = test_df['eki_name1'].fillna("NA")
test_df['eki_name2'] = test_df['eki_name2'].fillna("NA")
test_df['rosen_name1'] = test_df['rosen_name1'].fillna("NA")
test_df['rosen_name2'] = test_df['rosen_name2'].fillna("NA")
test_df['statuses'] = test_df['statuses'].fillna("NA")

In [54]:
# target_ymの経過年数に変換(先頭4文字を取得)
test_df['completion_year'] = test_df['year_built'].astype(float)
test_df['year_built'] = test_df['year_built'].fillna('199000').astype(str)
test_df['year_built'] = test_df['year_built'].astype(str).str[:4].astype(int)

test_df['target_ym'] = test_df['target_ym'].fillna('199000').astype(str)
test_df['year_built'] = test_df['target_ym'].astype(str).str[:4].astype(int) - test_df['year_built']

In [ ]:
# ■建物種別	
BUILDING_TYPE_DICT = {
    1: "マンション",
    2: "タウンハウス",
    3: "アパート",
    4: "一戸建",
    5: "テラスハウス",
    6: "土地",
    7: "駐車場",
    8: "ビル",
    9: "店舗",
    10: "倉庫",
    11: "工場",
    12: "寮",
    13: "ホテル",
    14: "旅館",
    15: "その他"
}

In [55]:
# /区切りのデータの変換
split = test_df["building_tag_id"].str.split("/")

# 辞書にあるキーだけを使って One-Hot 化
onehot = (
    split
    .apply(lambda xs: {TARGET_BUUIDLING_TAG_ID_DICT[x]: 1 for x in xs if x in TARGET_BUUIDLING_TAG_ID_DICT})
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
)
onehot = onehot.add_prefix("building_tag_id_")
onehot.head()

,building_tag_id_駐輪場あり,building_tag_id_プロパンガス,building_tag_id_タイル貼り,building_tag_id_宅配ボックス,building_tag_id_防犯カメラ,building_tag_id_オートロック,building_tag_id_ごみ出し24時間OK,building_tag_id_バイク置き場あり,building_tag_id_耐震構造,building_tag_id_24時間有人管理
0,1,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0


In [56]:
# /区切りのデータの変換
split = test_df["statuses"].str.split("/")

# 辞書にあるキーだけを使って One-Hot 化
onehot_statuses = (
    split
    .apply(lambda xs: {TARGET_STATUSES_TAG_DICT[x]: 1 for x in xs if x in TARGET_STATUSES_TAG_DICT})
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
)
onehot_statuses = onehot_statuses.add_prefix("statuses_")
onehot_statuses.head()

,statuses_クローゼット,statuses_バス・トイレ別,statuses_2階以上,statuses_バルコニー,statuses_温水洗浄便座,statuses_フローリング,statuses_エレベーター,statuses_TVモニタ付インターホン,statuses_エアコン,statuses_コンロ三口,statuses_追焚機能,statuses_カウンターキッチン,statuses_洗面所独立,statuses_ウォークインクローゼット,statuses_床暖房,statuses_室内洗濯機置場,statuses_出窓,statuses_最上階,statuses_ペット可
0,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0
3,1,0,0,1,1,0,0,1,0,0,1,1,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [57]:
# 結合
test_df = pd.concat([test_df, onehot, onehot_statuses], axis=1)
test_df.drop(['building_tag_id', 'statuses'], axis=1, inplace=True)
test_df.head()

,addr1_1,addr1_2,building_type,building_structure,bukken_id,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,land_kenpei,lat,lon,madori_kind_all,madori_number_all,money_kyoueki,money_kyoueki_tax,money_rimawari_now,park_distance,parking_distance,post1,room_kaisuu,rosen_name1,rosen_name2,school_ele_code,school_ele_distance,school_jun_code,school_jun_distance,target_ym,unit_area,year_built,walk_distance1,walk_distance2,completion_year,building_tag_id_駐輪場あり,building_tag_id_プロパンガス,building_tag_id_タイル貼り,building_tag_id_宅配ボックス,building_tag_id_防犯カメラ,building_tag_id_オートロック,building_tag_id_ごみ出し24時間OK,building_tag_id_バイク置き場あり,building_tag_id_耐震構造,building_tag_id_24時間有人管理,statuses_クローゼット,statuses_バス・トイレ別,statuses_2階以上,statuses_バルコニー,statuses_温水洗浄便座,statuses_フローリング,statuses_エレベーター,statuses_TVモニタ付インターホン,statuses_エアコン,statuses_コンロ三口,statuses_追焚機能,statuses_カウンターキッチン,statuses_洗面所独立,statuses_ウォークインクローゼット,statuses_床暖房,statuses_室内洗濯機置場,statuses_出窓,statuses_最上階,statuses_ペット可
0,24,205,1,5,50840,474.0,118.0,桑名,桑名,14,70,60.0,35.072193,136.688153,50,3,8400,2,NaN,NaN,0.0,511,4,JR関西本線,近鉄名古屋線,NaN,410.0,NaN,1387.0,202301,70.190002,28,887,909,199510.0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0
1,24,205,4,1,151717,NaN,NaN,馬道,NA,2,171,60.0,35.066061,136.673603,50,6,0,3,NaN,NaN,NaN,511,0,三岐鉄道北勢線,NA,NaN,1030.0,NaN,1361.0,202301,171.820007,31,880,9999,199206.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,23,224,4,10,167483,650.0,NaN,大野町,NA,2,78,60.0,34.937964,136.854324,50,3,0,3,NaN,210.0,NaN,478,0,名鉄常滑線,NA,NaN,750.0,NaN,900.0,202301,92.129997,48,2800,9999,197511.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0
3,23,224,4,1,210231,460.0,1250.0,寺本,NA,1,93,70.0,35.003429,136.877587,50,4,0,0,NaN,NaN,NaN,478,0,名鉄常滑線,NA,NaN,1400.0,NaN,1800.0,202301,93.139999,7,640,9999,201603.0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,1,1,1,0,0,0,0,0,0
4,24,205,4,1,391649,522.0,NaN,星川,七和,2,105,50.0,35.050090,136.627432,50,4,0,3,NaN,2721.0,NaN,511,0,三岐鉄道北勢線,三岐鉄道北勢線,NaN,1581.0,NaN,1736.0,202301,105.980003,29,2160,2400,199411.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [58]:
# コードを文字列に変換
test_df["building_type"] = test_df["building_type"].map(BUILDING_TYPE_DICT).fillna("不明")
test_df["building_structure"] = test_df["building_structure"].map(BUILDING_STURUCTURE_DICT).fillna("不明")
test_df["madori_kind_all"] = test_df["madori_kind_all"].map(MADORI_KIND_ALL_DCT).fillna("不明")

In [61]:
RADIUS_METER = 800  # 800メートル(徒歩10分)

# 1. dfからGeoDataFrameを作成 (WGS84を指定)
gdf_base = gpd.GeoDataFrame(
    test_df.copy(),
    geometry=gpd.points_from_xy(test_df['lon'], test_df['lat']),
    crs="EPSG:4326"
)

# 2. メートル単位の座標系に変換 (日本近海用: EPSG:6677)
# 800mという距離を正確に扱うために必要です
gdf_base_m = gdf_base.to_crs(epsg=6677)
gdf_station_m = gdf_station.to_crs(epsg=6677)

# 3. 各地点を中心に800mのバッファ（円）を作成
# geometryをPointからPolygon(円)に置き換えます
gdf_base_m['geometry'] = gdf_base_m.geometry.buffer(RADIUS_METER)

# 4. 空間結合 (sjoin)
# 800m圏内(円)に含まれる駅を紐付けます
# left joinにすることで、駅がない地点も保持します
joined = gpd.sjoin(
    gdf_base_m,
    gdf_station_m[['駅名', '乗降客数2023', 'geometry']],
    how='left',
    predicate='intersects'
)

# 5. 特徴量の集計
# 地点ごとに「駅の数」と「乗降客数の合計」を計算
stats = joined.groupby(joined.index).agg({
    '駅名': 'count',           # 駅の数
    '乗降客数2023': 'sum'      # 乗降客数の合計
}).rename(columns={
    '駅名': f'station_count_{RADIUS_METER}m',
    '乗降客数2023': f'total_passengers_{RADIUS_METER}m'
})

# 6. 元のdfに特徴量を結合
test_df = test_df.merge(stats, left_index=True, right_index=True)

# 結果の確認
test_df.head()

,addr1_1,addr1_2,building_type,building_structure,bukken_id,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,land_kenpei,lat,lon,madori_kind_all,madori_number_all,money_kyoueki,money_kyoueki_tax,money_rimawari_now,park_distance,parking_distance,post1,room_kaisuu,rosen_name1,rosen_name2,school_ele_code,school_ele_distance,school_jun_code,school_jun_distance,target_ym,unit_area,year_built,walk_distance1,walk_distance2,completion_year,building_tag_id_駐輪場あり,building_tag_id_プロパンガス,building_tag_id_タイル貼り,building_tag_id_宅配ボックス,building_tag_id_防犯カメラ,building_tag_id_オートロック,building_tag_id_ごみ出し24時間OK,building_tag_id_バイク置き場あり,building_tag_id_耐震構造,building_tag_id_24時間有人管理,statuses_クローゼット,statuses_バス・トイレ別,statuses_2階以上,statuses_バルコニー,statuses_温水洗浄便座,statuses_フローリング,statuses_エレベーター,statuses_TVモニタ付インターホン,statuses_エアコン,statuses_コンロ三口,statuses_追焚機能,statuses_カウンターキッチン,statuses_洗面所独立,statuses_ウォークインクローゼット,statuses_床暖房,statuses_室内洗濯機置場,statuses_出窓,statuses_最上階,statuses_ペット可,station_count_800m,total_passengers_800m
0,24,205,マンション,SRC,50840,474.0,118.0,桑名,桑名,14,70,60.0,35.072193,136.688153,LDK,3,8400,2,NaN,NaN,0.0,511,4,JR関西本線,近鉄名古屋線,NaN,410.0,NaN,1387.0,202301,70.190002,28,887,909,199510.0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,3,31505.0
1,24,205,一戸建,木造,151717,NaN,NaN,馬道,NA,2,171,60.0,35.066061,136.673603,LDK,6,0,3,NaN,NaN,NaN,511,0,三岐鉄道北勢線,NA,NaN,1030.0,NaN,1361.0,202301,171.820007,31,880,9999,199206.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,547.0
2,23,224,一戸建,軽量鉄骨,167483,650.0,NaN,大野町,NA,2,78,60.0,34.937964,136.854324,LDK,3,0,3,NaN,210.0,NaN,478,0,名鉄常滑線,NA,NaN,750.0,NaN,900.0,202301,92.129997,48,2800,9999,197511.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0.0
3,23,224,一戸建,木造,210231,460.0,1250.0,寺本,NA,1,93,70.0,35.003429,136.877587,LDK,4,0,0,NaN,NaN,NaN,478,0,名鉄常滑線,NA,NaN,1400.0,NaN,1800.0,202301,93.139999,7,640,9999,201603.0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,1,1,1,0,0,0,0,0,0,1,3988.0
4,24,205,一戸建,木造,391649,522.0,NaN,星川,七和,2,105,50.0,35.050090,136.627432,LDK,4,0,3,NaN,2721.0,NaN,511,0,三岐鉄道北勢線,三岐鉄道北勢線,NaN,1581.0,NaN,1736.0,202301,105.980003,29,2160,2400,199411.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0


In [62]:
RADIUS_METER = 400  # 400メートル(徒歩5分)

# 1. dfからGeoDataFrameを作成 (WGS84を指定)
gdf_base = gpd.GeoDataFrame(
    test_df.copy(),
    geometry=gpd.points_from_xy(test_df['lon'], test_df['lat']),
    crs="EPSG:4326"
)

# 2. メートル単位の座標系に変換 (日本近海用: EPSG:6677)
# 800mという距離を正確に扱うために必要です
gdf_base_m = gdf_base.to_crs(epsg=6677)
gdf_station_m = gdf_station.to_crs(epsg=6677)

# 3. 各地点を中心に800mのバッファ（円）を作成
# geometryをPointからPolygon(円)に置き換えます
gdf_base_m['geometry'] = gdf_base_m.geometry.buffer(RADIUS_METER)

# 4. 空間結合 (sjoin)
# 800m圏内(円)に含まれる駅を紐付けます
# left joinにすることで、駅がない地点も保持します
joined = gpd.sjoin(
    gdf_base_m,
    gdf_station_m[['駅名', '乗降客数2023', 'geometry']],
    how='left',
    predicate='intersects'
)

# 5. 特徴量の集計
# 地点ごとに「駅の数」と「乗降客数の合計」を計算
stats = joined.groupby(joined.index).agg({
    '駅名': 'count',           # 駅の数
    '乗降客数2023': 'sum'      # 乗降客数の合計
}).rename(columns={
    '駅名': f'station_count_{RADIUS_METER}m',
    '乗降客数2023': f'total_passengers_{RADIUS_METER}m'
})

# 6. 元のdfに特徴量を結合
test_df_combined_station = test_df.merge(stats, left_index=True, right_index=True)

# 結果の確認
test_df_combined_station.head()

,addr1_1,addr1_2,building_type,building_structure,bukken_id,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,land_kenpei,lat,lon,madori_kind_all,madori_number_all,money_kyoueki,money_kyoueki_tax,money_rimawari_now,park_distance,parking_distance,post1,room_kaisuu,rosen_name1,rosen_name2,school_ele_code,school_ele_distance,school_jun_code,school_jun_distance,target_ym,unit_area,year_built,walk_distance1,walk_distance2,completion_year,building_tag_id_駐輪場あり,building_tag_id_プロパンガス,building_tag_id_タイル貼り,building_tag_id_宅配ボックス,building_tag_id_防犯カメラ,building_tag_id_オートロック,building_tag_id_ごみ出し24時間OK,building_tag_id_バイク置き場あり,building_tag_id_耐震構造,building_tag_id_24時間有人管理,statuses_クローゼット,statuses_バス・トイレ別,statuses_2階以上,statuses_バルコニー,statuses_温水洗浄便座,statuses_フローリング,statuses_エレベーター,statuses_TVモニタ付インターホン,statuses_エアコン,statuses_コンロ三口,statuses_追焚機能,statuses_カウンターキッチン,statuses_洗面所独立,statuses_ウォークインクローゼット,statuses_床暖房,statuses_室内洗濯機置場,statuses_出窓,statuses_最上階,statuses_ペット可,station_count_800m,total_passengers_800m,station_count_400m,total_passengers_400m
0,24,205,マンション,SRC,50840,474.0,118.0,桑名,桑名,14,70,60.0,35.072193,136.688153,LDK,3,8400,2,NaN,NaN,0.0,511,4,JR関西本線,近鉄名古屋線,NaN,410.0,NaN,1387.0,202301,70.190002,28,887,909,199510.0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,3,31505.0,0,0.0
1,24,205,一戸建,木造,151717,NaN,NaN,馬道,NA,2,171,60.0,35.066061,136.673603,LDK,6,0,3,NaN,NaN,NaN,511,0,三岐鉄道北勢線,NA,NaN,1030.0,NaN,1361.0,202301,171.820007,31,880,9999,199206.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,547.0,0,0.0
2,23,224,一戸建,軽量鉄骨,167483,650.0,NaN,大野町,NA,2,78,60.0,34.937964,136.854324,LDK,3,0,3,NaN,210.0,NaN,478,0,名鉄常滑線,NA,NaN,750.0,NaN,900.0,202301,92.129997,48,2800,9999,197511.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0.0,0,0.0
3,23,224,一戸建,木造,210231,460.0,1250.0,寺本,NA,1,93,70.0,35.003429,136.877587,LDK,4,0,0,NaN,NaN,NaN,478,0,名鉄常滑線,NA,NaN,1400.0,NaN,1800.0,202301,93.139999,7,640,9999,201603.0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,1,0,0,1,1,1,0,0,0,0,0,0,1,3988.0,1,3988.0
4,24,205,一戸建,木造,391649,522.0,NaN,星川,七和,2,105,50.0,35.050090,136.627432,LDK,4,0,3,NaN,2721.0,NaN,511,0,三岐鉄道北勢線,三岐鉄道北勢線,NaN,1581.0,NaN,1736.0,202301,105.980003,29,2160,2400,199411.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0.0


In [ ]:
train_df_combined_station.shape

(313664, 68)

In [63]:
test_df_combined_station.shape

(112437, 68)

In [64]:
test_df_combined_station["rosen_name1_is_JR"] = test_df_combined_station["rosen_name1"].apply(lambda x: 1 if x.startswith("JR") and x.endswith("線") else 0)
test_df_combined_station["rosen_name2_is_JR"] = test_df_combined_station["rosen_name2"].apply(lambda x: 1 if x.startswith("JR") and x.endswith("線") else 0)
test_df_combined_station[test_df_combined_station["rosen_name1_is_JR"] == 1].head()

,addr1_1,addr1_2,building_type,building_structure,bukken_id,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,land_kenpei,lat,lon,madori_kind_all,madori_number_all,money_kyoueki,money_kyoueki_tax,money_rimawari_now,park_distance,parking_distance,post1,room_kaisuu,rosen_name1,rosen_name2,school_ele_code,school_ele_distance,school_jun_code,school_jun_distance,target_ym,unit_area,year_built,walk_distance1,walk_distance2,completion_year,building_tag_id_駐輪場あり,building_tag_id_プロパンガス,building_tag_id_タイル貼り,building_tag_id_宅配ボックス,building_tag_id_防犯カメラ,building_tag_id_オートロック,building_tag_id_ごみ出し24時間OK,building_tag_id_バイク置き場あり,building_tag_id_耐震構造,building_tag_id_24時間有人管理,statuses_クローゼット,statuses_バス・トイレ別,statuses_2階以上,statuses_バルコニー,statuses_温水洗浄便座,statuses_フローリング,statuses_エレベーター,statuses_TVモニタ付インターホン,statuses_エアコン,statuses_コンロ三口,statuses_追焚機能,statuses_カウンターキッチン,statuses_洗面所独立,statuses_ウォークインクローゼット,statuses_床暖房,statuses_室内洗濯機置場,statuses_出窓,statuses_最上階,statuses_ペット可,station_count_800m,total_passengers_800m,station_count_400m,total_passengers_400m,rosen_name1_is_JR,rosen_name2_is_JR
0,24,205,マンション,SRC,50840,474.0,118.0,桑名,桑名,14,70,60.0,35.072193,136.688153,LDK,3,8400,2,NaN,NaN,0.0,511,4,JR関西本線,近鉄名古屋線,NaN,410.0,NaN,1387.0,202301,70.190002,28,887,909,199510.0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,3,31505.0,0,0.0,1,0
12,25,214,一戸建,木造,123860,858.0,600.0,米原,フジテック前,2,138,70.0,35.311584,136.282109,DK,6,0,3,NaN,NaN,NaN,521,0,JR東海道・山陽本線,近江鉄道近江本線,NaN,528.0,NaN,273.0,202301,138.919998,47,1200,2400,197604.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,22624.0,0,0.0,1,0
13,25,214,不明,不明,374543,NaN,NaN,柏原,近江長岡,1,160,NaN,35.343238,136.404917,LDK,6,0,0,NaN,NaN,NaN,521,0,JR東海道本線,JR東海道本線,NaN,500.0,NaN,1100.0,202301,NaN,33,640,240,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,424.0,1,424.0,1,1
14,25,214,一戸建,木造,96493,NaN,NaN,近江長岡,NA,2,101,70.0,35.397991,136.347296,LDK,4,0,3,NaN,NaN,NaN,521,0,JR東海道本線,NA,NaN,2000.0,NaN,3100.0,202301,101.519997,35,5600,9999,198811.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,1,0,0,0,0,0,1,0,0,0,0.0,0,0.0,1,0
15,40,230,一戸建,木造,125266,NaN,NaN,波多江,糸島高校前,1,59,60.0,33.563095,130.220809,DK,3,0,2,NaN,NaN,0.0,819,0,JR筑肥線,JR筑肥線,NaN,600.0,NaN,2500.0,202301,106.089996,0,800,880,202307.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,2,9400.0,0,0.0,1,1


In [65]:
test_df_combined_station.to_pickle(f"{ROOT_DIR}/0107_preprocessed_test_df.pkl")

### 公示価格の追加

In [3]:
from tqdm import tqdm

In [4]:
# データの読み込み
ROOT_DIR = "../dataset"

train_df_combined_station = pd.read_pickle(f"{ROOT_DIR}/0108_preprocessed_train_df.pkl")
test_df_combined_station = pd.read_pickle(f"{ROOT_DIR}/0108_preprocessed_test_df.pkl")

In [5]:
# ドライブに保存してある国土数値情報をデータフレームにする
gdf_land_values_2023 = gpd.read_file(f'{ROOT_DIR}/L01-23_GML')
gdf_land_values_2022 = gpd.read_file(f'{ROOT_DIR}/L01-22_GML')

target_columns_2023 = ['L01_001', 'L01_002', 'L01_005','L01_006','L01_007','L01_022','geometry']
rename_mapping_2023 = {'L01_001':'見出し番号', 'L01_002':'一連番号', 'L01_005':'年度','L01_006':'公示価格','L01_007':'対前年変動率','L01_022':'標準地行政区域コード'}

gdf_land_values_2023 = gdf_land_values_2023[target_columns_2023]
gdf_land_values_2022 = gdf_land_values_2022[target_columns_2023]

gdf_land_values_2023 = gdf_land_values_2023.rename(columns=rename_mapping_2023)
gdf_land_values_2022 = gdf_land_values_2022.rename(columns=rename_mapping_2023)
gdf_land_values_2022.head()

,見出し番号,一連番号,年度,公示価格,対前年変動率,標準地行政区域コード,geometry
0,000,001,2022,397000,3.4,01101,POINT (141.31404 43.05545)
1,000,002,2022,170000,8.3,01101,POINT (141.3394 43.04181)
2,000,003,2022,160000,8.1,01101,POINT (141.34456 43.04713)
3,000,004,2022,186000,8.8,01101,POINT (141.34944 43.03636)
4,000,005,2022,52300,4.6,01101,POINT (141.3153 43.0439)


In [6]:
# 一連番号の加工
gdf_land_values_2022['一連番号'] = gdf_land_values_2022['見出し番号'].astype(str) + "-" + gdf_land_values_2022['一連番号'].astype(str)
gdf_land_values_2023['一連番号'] = gdf_land_values_2023['見出し番号'].astype(str) + "-" + gdf_land_values_2023['一連番号'].astype(str)

gdf_land_values_2022.drop(["見出し番号"], axis=1, inplace=True)
gdf_land_values_2023.drop(["見出し番号"], axis=1, inplace=True)

In [7]:
# 公示価格カラムを数値型に変換
gdf_land_values_2022['公示価格'] = pd.to_numeric(gdf_land_values_2022['公示価格'], errors='coerce')
gdf_land_values_2023['公示価格'] = pd.to_numeric(gdf_land_values_2023['公示価格'], errors='coerce')
gdf_land_values_2022['年度'] = gdf_land_values_2022['年度'].astype(int)
gdf_land_values_2023['年度'] = gdf_land_values_2023['年度'].astype(int)

In [8]:
gdf_land_values_2022.head()

,一連番号,年度,公示価格,対前年変動率,標準地行政区域コード,geometry
0,000-001,2022,397000,3.4,01101,POINT (141.31404 43.05545)
1,000-002,2022,170000,8.3,01101,POINT (141.3394 43.04181)
2,000-003,2022,160000,8.1,01101,POINT (141.34456 43.04713)
3,000-004,2022,186000,8.8,01101,POINT (141.34944 43.03636)
4,000-005,2022,52300,4.6,01101,POINT (141.3153 43.0439)


In [9]:
from scipy.spatial import cKDTree


# 1. dfからGeoDataFrameを作成 (WGS84を指定)
gdf_base = gpd.GeoDataFrame(
    train_df_combined_station.copy(),
    geometry=gpd.points_from_xy(train_df_combined_station['lon'], train_df_combined_station['lat']),
    crs="EPSG:4326"
)

# 2. メートル単位の座標系に変換 (日本近海用: EPSG:6677)
# 800mという距離を正確に扱うために必要です
gdf_base_m = gdf_base.to_crs(epsg=6677)
gdf_land_values_m = gdf_land_values_2022.to_crs(epsg=6677)

# b_df の座標
land_place_coords = np.vstack([
    gdf_land_values_m.geometry.x,
    gdf_land_values_m.geometry.y
]).T

tree = cKDTree(land_place_coords)

k = 5

value_medians = []
ratio_means = []

for geom in tqdm(gdf_base_m.geometry):
    x, y = geom.x, geom.y

    # 最近傍5点のインデックス取得
    distances, indices = tree.query([x, y], k=k)

    nearest = gdf_land_values_m.iloc[indices]

    value_medians.append(nearest["公示価格"].median())
    ratio_means.append(nearest["対前年変動率"].mean())


train_df_combined_station['land_value_median_5nn'] = value_medians
train_df_combined_station['land_value_ratio_mean_5nn'] = ratio_means

100%|██████████| 313664/313664 [04:47<00:00, 1091.87it/s]


In [10]:
from scipy.spatial import cKDTree


# 1. dfからGeoDataFrameを作成 (WGS84を指定)
gdf_base = gpd.GeoDataFrame(
    test_df_combined_station.copy(),
    geometry=gpd.points_from_xy(test_df_combined_station['lon'], test_df_combined_station['lat']),
    crs="EPSG:4326"
)

# 2. メートル単位の座標系に変換 (日本近海用: EPSG:6677)
# 800mという距離を正確に扱うために必要です
gdf_base_m = gdf_base.to_crs(epsg=6677)
gdf_land_values_m = gdf_land_values_2023.to_crs(epsg=6677)

# b_df の座標
land_place_coords = np.vstack([
    gdf_land_values_m.geometry.x,
    gdf_land_values_m.geometry.y
]).T

tree = cKDTree(land_place_coords)

k = 5

value_medians = []
ratio_means = []

for geom in tqdm(gdf_base_m.geometry):
    x, y = geom.x, geom.y

    # 最近傍5点のインデックス取得
    distances, indices = tree.query([x, y], k=k)

    nearest = gdf_land_values_m.iloc[indices]

    value_medians.append(nearest["公示価格"].median())
    ratio_means.append(nearest["対前年変動率"].mean())


test_df_combined_station['land_value_median_5nn'] = value_medians
test_df_combined_station['land_value_ratio_mean_5nn'] = ratio_means

100%|██████████| 112437/112437 [01:17<00:00, 1457.35it/s]


In [11]:
train_df_combined_station.shape, test_df_combined_station.shape

((313664, 42), (112437, 41))

In [12]:
train_df_combined_station.head()

,addr1_1,addr1_2,building_type,building_structure,bukken_id,convenience_distance,drugstore_distance,eki_name1,eki_name2,floor_count,house_area,lat,lon,madori_kind_all,madori_number_all,money_kyoueki,money_kyoueki_tax,money_rimawari_now,park_distance,parking_distance,post1,post2,room_kaisuu,rosen_name1,rosen_name2,school_ele_distance,school_jun_distance,target_ym,unit_area,year_built,walk_distance1,walk_distance2,money_room,target_y,madori_str,post_number,station_count_800m,total_passengers_800m,station_count_1600m,total_passengers_1600m,land_value_median_5nn,land_value_ratio_mean_5nn
0,24,205,一戸建,木造,360840,NaN,NaN,在良,NA,2,106.82,35.047688,136.637467,LDK,4,0,3,NaN,NaN,NaN,511,0932,0,三岐鉄道北勢線,NA,2000.0,2000.0,201901,106.820000,27,1840,9999,13980000,2019,4LDK,511-0932,0,0.0,1,231.0,41300.0,-0.20
1,24,205,一戸建,軽量鉄骨,267022,NaN,NaN,星川,NA,2,134.04,35.074625,136.639936,LDK,4,0,3,NaN,NaN,NaN,511,0902,0,三岐鉄道北勢線,NA,350.0,1100.0,201901,134.039993,38,1920,9999,24480000,2019,4LDK,511-0902,0,0.0,2,1308.0,57500.0,0.32
2,24,205,一戸建,木造,194544,NaN,NaN,蓮花寺,NA,2,114.59,35.072248,136.644708,LDK,4,0,3,NaN,NaN,NaN,511,0902,0,三岐鉄道北勢線,NA,850.0,1200.0,201901,114.589996,24,2000,9999,24480000,2019,4LDK,511-0902,0,0.0,3,1848.0,63400.0,0.26
3,23,224,一戸建,木造,345772,NaN,NaN,寺本,尾張横須賀,2,106.81,35.003174,136.875602,LDK,3,0,2,NaN,NaN,0.0,478,0001,0,名鉄常滑線,名鉄常滑線,1400.0,1400.0,201901,106.809998,17,480,1600,16300000,2019,3LDK,478-0001,1,3988.0,3,15129.0,66700.0,0.96
4,23,224,マンション,RC,172718,NaN,1060.0,寺本,朝倉,6,76.74,35.002403,136.875754,LDK,3,8440,2,NaN,NaN,0.0,478,0001,1,名鉄常滑線,名鉄常滑線,1080.0,1410.0,201901,76.739998,12,320,1600,18800000,2019,3LDK,478-0001,1,3988.0,3,15129.0,66700.0,0.96


#### ボツ案

In [ ]:
RADIUS_METER_LIST = [300, 500, 1000, 2000]

# 1. dfからGeoDataFrameを作成 (WGS84を指定)
gdf_base = gpd.GeoDataFrame(
    train_df_combined_station.copy(),
    geometry=gpd.points_from_xy(train_df_combined_station['lon'], train_df_combined_station['lat']),
    crs="EPSG:4326"
)

# 2. メートル単位の座標系に変換 (日本近海用: EPSG:6677)
# 800mという距離を正確に扱うために必要です
gdf_base_m = gdf_base.to_crs(epsg=6677)
gdf_land_values_m = gdf_land_values_2022.to_crs(epsg=6677)

stats_list = []
for radius_m in tqdm(RADIUS_METER_LIST):
    # 3. 各地点を中心に800mのバッファ（円）を作成
    # geometryをPointからPolygon(円)に置き換えます
    gdf_base_m['geometry'] = gdf_base_m.geometry.buffer(radius_m)
    # 4. 空間結合 (sjoin)
    # 800m圏内(円)に含まれる駅を紐付けます
    # left joinにすることで、駅がない地点も保持します
    joined = gpd.sjoin(
        gdf_base_m,
        gdf_land_values_m[['一連番号', '公示価格', '対前年変動率', 'geometry']],
        how='left',
        predicate='intersects'
    )

    # 5. 特徴量の集計
    # 地点ごとに「土地価格の中央値」と「対前年変動率の平均」を計算
    stats = joined.groupby(joined.index).agg({
        '一連番号': 'count',           # 地点の数
        '公示価格': 'median',      # 土地価格の中央値
        '対前年変動率': 'mean',      # 対前年変動率の平均
    }).rename(columns={
        '一連番号': f'point_count_{radius_m}m',
        '公示価格': f'land_price_median_{radius_m}m',
        '対前年変動率': f'price_change_mean_{radius_m}m'
    })

    stats_list.append(stats)

stats = pd.concat(stats_list, axis=1)
# # 6. 元のdfに特徴量を結合
# test_df = test_df.merge(stats, left_index=True, right_index=True)

# # 結果の確認
# test_df.head()

 75%|███████▌  | 3/4 [07:35<03:12, 192.23s/it]

In [ ]:
radius = 1000  # meters

value_weighted_means = []
ratio_weighted_means = []

for geom in tqdm(gdf_base_m.geometry):
    buffer = geom.buffer(radius)

    nearby = gdf_land_values_m[gdf_land_values_m.intersects(buffer)].copy()

    if nearby.empty:
        value_weighted_means.append(np.nan)
        ratio_weighted_means.append(np.nan)
        continue

    # 距離（a_df の点から b_df の各点）
    distances = nearby.geometry.distance(geom)

    # ゼロ距離対策（完全一致）
    distances = distances.replace(0, 1e-6)

    # 重み（距離の逆数）
    weights = 1 / distances

    # 重み付き平均
    value_wmean = np.average(nearby["公示価格"], weights=weights)
    ratio_wmean = np.average(nearby["対前年変動率"], weights=weights)

    value_weighted_means.append(value_wmean)
    ratio_weighted_means.append(ratio_wmean)


KeyboardInterrupt: 

In [25]:
stats["point_count_1000m"].value_counts()

point_count_1000m
4     33350
3     31859
2     31834
5     30515
1     28753
6     28049
7     24421
0     23892
8     19684
9     15166
10    10729
11     7356
12     6156
13     4756
14     3749
15     3021
16     2129
17     1756
18     1397
19      984
20      973
21      687
22      632
23      442
24      322
25      224
26      221
28      162
27      144
29      121
30       63
31       38
32       28
33       27
34       18
35        6
Name: count, dtype: int64

In [24]:
tmp = train_df_combined_station.merge(stats, left_index=True, right_index=True)
tmp["money_room_per_area"] = tmp["money_room"] / tmp["house_area"]
tmp.loc[:, ["money_room_per_area", "land_price_median_1000m"]]

,money_room_per_area,land_price_median_1000m
0,130874.368096,31550.0
1,182632.050134,63400.0
2,213631.206912,56100.0
3,152607.433761,69100.0
4,244983.059682,69100.0
...,...,...
313659,213432.835821,72400.0
313660,239240.506329,80950.0
313661,223943.661972,78200.0
313662,262637.362637,41300.0


In [31]:
tmp["5_nearest_land_price_median"] = value_medians
tmp.loc[:, ["money_room_per_area", "point_count_1000m", "land_price_median_1000m", "5_nearest_land_price_median"]]

,money_room_per_area,point_count_1000m,land_price_median_1000m,5_nearest_land_price_median
0,130874.368096,2,31550.0,41300.0
1,182632.050134,3,63400.0,57500.0
2,213631.206912,3,56100.0,63400.0
3,152607.433761,6,69100.0,66700.0
4,244983.059682,6,69100.0,66700.0
...,...,...,...,...
313659,213432.835821,9,72400.0,72400.0
313660,239240.506329,10,80950.0,84200.0
313661,223943.661972,9,78200.0,83700.0
313662,262637.362637,2,41300.0,45600.0


## データの保存

In [13]:
import pickle


with open(f"{ROOT_DIR}/0109_preprocessed_train_df.pkl", "wb") as f:
    pickle.dump(train_df_combined_station, f)

with open(f"{ROOT_DIR}/0109_preprocessed_test_df.pkl", "wb") as f:
    pickle.dump(test_df_combined_station, f)

In [8]:
# pickleをparqetに変換
PREFIX = "0109_preprocessed"
train_df = pd.read_pickle(f"{ROOT_DIR}/{PREFIX}_train_df.pkl")
train_df.to_parquet(f"{ROOT_DIR}/{PREFIX}_train_df.parquet")

In [9]:
# pickleをparqetに変換
test_df = pd.read_pickle(f"{ROOT_DIR}/{PREFIX}_test_df.pkl")
test_df.to_parquet(f"{ROOT_DIR}/{PREFIX}_test_df.parquet")

# やりたいこと
- 郵便番号の組み合わせ
- 近傍点の地下公示価格
- 

In [20]:
train_money_room = train_df.money_room.values
test_id = test_df['id']

train_df.drop(['money_room'], axis=1, inplace=True)
test_df.drop(['id'], axis=1, inplace=True)

In [17]:
train_df[train_df["bukken_id"].isin(dup_bukken_list)].loc[:, ["prefecture_name", "city_town_village_name", "bukken_id", "target_ym", "year_built", "building_age", "money_room"]].sort_values(["bukken_id", "target_ym", "city_town_village_name"])

,prefecture_name,city_town_village_name,bukken_id,target_ym,year_built,building_age,money_room
1538,東京都,大田区,100472,201901,1996,23.0,26800000
38451,東京都,大田区,100472,201907,1996,23.0,25800000
84439,東京都,大田区,100472,202001,1996,24.0,24800000
42830,東京都,世田谷区,101333,201907,2005,14.0,76800000
89281,東京都,世田谷区,101333,202001,2005,15.0,74800000
...,...,...,...,...,...,...,...
84659,東京都,大田区,99642,202001,1984,36.0,24900000
136310,東京都,江東区,99668,202007,2014,6.0,22500000
188406,東京都,江東区,99668,202101,2014,7.0,21000000
214607,東京都,足立区,99705,202101,1981,40.0,16800000


## 欠損データ

In [18]:
train_missing = get_missing_report(train_df)
test_missing = get_missing_report(test_df)

,column_name,missing_count,missing_ratio
8,building_name_ruby,363924,1.0
47,name_ruby,363924,1.0
131,school_ele_code,363924,1.0
134,school_jun_code,363924,1.0
146,money_hoshou_company,363924,1.0
147,free_rent_duration,363924,1.0
148,free_rent_gen_timing,363924,1.0


### 欠損値を持つ列が多数あり、中には完全に空のものもあります。今後処理する必要があります。

In [21]:
numerical_columns = [
    "unit_count",
    "lon",
    "lat",
    "total_floor_area",
    "building_area",
    "floor_count",
    "basement_floor_count",
    "building_land_area",
    "land_area_all",
    "unit_area_min",
    "unit_area_max",
    "land_setback",
    "land_kenpei",
    "land_youseki",
    "room_floor",
    "balcony_area",
    "room_count",
    "unit_area",
    "empty_number",
    "nl",
    "el",
    "bus_time1",
    "walk_distance1",
    "bus_time2",
    "walk_distance2",
    "traffic_car",
    "snapshot_land_area",
    "snapshot_land_shidou",
    "land_shidou_a",
    "land_shidou_b",
    "land_mochibun_a",
    "land_mochibun_b",
    "house_area",
    "room_kaisuu",
    "madori_number_all",
    "money_kyoueki",
    "money_rimawari_now",
    "money_shuuzen",
    "money_shuuzenkikin",
    "money_sonota1",
    "money_sonota2",
    "money_sonota3",
    "parking_money",
    "parking_distance",
    "parking_number",
    "school_ele_distance",
    "school_jun_distance",
    "convenience_distance",
    "super_distance",
    "hospital_distance",
    "park_distance",
    "drugstore_distance",
    "bank_distance",
    "shopping_street_distance",
    "est_other_distance",
]

In [22]:
# カテゴリカル変数を文字列型に変更
train_df = train_df.fillna(np.nan)
test_df = test_df.fillna(np.nan)

categorical_cols = [c for c in train_df.columns if c not in numerical_columns]

for c in categorical_cols:
    train_df[c] = train_df[c].astype(str)
    test_df[c] = test_df[c].astype(str)

## テストデータの分析

In [29]:
train_df = train_df.to_pandas(use_pyarrow_extension_array=True)
test_df = test_df.to_pandas(use_pyarrow_extension_array=True)

In [53]:
train_df.loc[:, ['prefecture_name', 'city_town_village_name']].value_counts().head(50) / train_df.shape[0]

prefecture_name  city_town_village_name
東京都              世田谷区                      0.010950
                 足立区                       0.009510
埼玉県              川口市                       0.009159
東京都              八王子市                      0.008516
                 板橋区                       0.008510
                 大田区                       0.008430
                 江東区                       0.008279
                 新宿区                       0.007661
                 江戸川区                      0.007515
千葉県              船橋市                       0.007191
東京都              港区                        0.007125
                 練馬区                       0.007059
千葉県              松戸市                       0.006499
東京都              杉並区                       0.006463
大阪府              枚方市                       0.006353
東京都              葛飾区                       0.006249
和歌山県             和歌山市                      0.006031
滋賀県              大津市                       0.005977
神奈川県             横須賀市   

In [30]:
test_df.loc[:, ['prefecture_name', 'city_town_village_name']].value_counts().head(50) / test_df.shape[0]

prefecture_name  city_town_village_name
東京都              世田谷区                      0.010931
                 足立区                       0.009872
埼玉県              川口市                       0.008912
東京都              大田区                       0.008769
                 江東区                       0.008698
                 板橋区                       0.008165
千葉県              船橋市                       0.008102
大阪府              枚方市                       0.008031
東京都              新宿区                       0.007862
大阪府              東大阪市                      0.007675
東京都              江戸川区                      0.007604
                 港区                        0.007560
                 八王子市                      0.007542
兵庫県              姫路市                       0.007498
東京都              練馬区                       0.007444
兵庫県              西宮市                       0.006670
東京都              品川区                       0.006421
千葉県              市川市                       0.006146
東京都              杉並区    

In [31]:
train_df["target_ym"].value_counts() / train_df.shape[0]

target_ym
202007    0.144299
202001    0.141381
202207    0.127672
201907    0.125782
202101    0.124913
202201     0.11844
202107    0.117173
201901     0.10034
Name: count, dtype: double[pyarrow]

In [32]:
tmp = train_df[(train_df["prefecture_name"].isin(["東京都", "埼玉県"])) & (train_df["city_town_village_name"].isin(["世田谷区", "足立区", "川口市", "大田区", "江東区"]))].groupby("bukken_id")["target_ym"].count().reset_index(level=0)
dup_bukken_list = tmp[tmp["target_ym"] >= 2]["bukken_id"]

del tmp

In [47]:
len(set(dup_bukken_list.to_list()))

1271

In [37]:
train_df["year_built"].head()

0    199204.0
1    198108.0
2    199506.0
3    200203.0
4    200703.0
Name: year_built, dtype: large_string[pyarrow]

In [43]:
# 築年経過月の追加
def calculate_month_difference(target_ym, year_built):
    try:
        target_year = int(target_ym[:4])
        target_month = int(float(target_ym[4:]))
        built_year = int(year_built[:4])
        built_month = int(float(year_built[4:]))

        return (target_year - built_year) * 12 + target_month - built_month
    except:
        return np.nan

train_df["building_age_month"] = train_df.apply(
    lambda row: calculate_month_difference(row["target_ym"], row["year_built"]), axis=1
)

In [51]:
train_df.drop("money_room", inplace=True, axis=1)
train_df = pd.concat([train_df, pd.DataFrame(train_money_room, columns=["money_room"])], axis=1)
train_df[train_df["bukken_id"].isin(dup_bukken_list)].loc[:, ["prefecture_name", "city_town_village_name", "bukken_id", "target_ym", "year_built", "building_age_month", "money_room"]].sort_values(["bukken_id", "target_ym", "city_town_village_name"])

,prefecture_name,city_town_village_name,bukken_id,target_ym,year_built,building_age_month,money_room
1538,東京都,大田区,100472,201901,199601.0,276.0,26800000
38451,東京都,大田区,100472,201907,199601.0,282.0,25800000
84439,東京都,大田区,100472,202001,199601.0,288.0,24800000
42830,東京都,世田谷区,101333,201907,200503.0,172.0,76800000
89281,東京都,世田谷区,101333,202001,200503.0,178.0,74800000
...,...,...,...,...,...,...,...
84659,東京都,大田区,99642,202001,198410.0,423.0,24900000
136310,東京都,江東区,99668,202007,201408.0,71.0,22500000
188406,東京都,江東区,99668,202101,201408.0,77.0,21000000
214607,東京都,足立区,99705,202101,198102.0,479.0,16800000


In [54]:
train_df["target_ym"].unique()

<ArrowExtensionArray>
['201901', '201907', '202001', '202007', '202101', '202107', '202201',
 '202207']
Length: 8, dtype: large_string[pyarrow]

In [55]:
test_df["target_ym"].unique()

<ArrowExtensionArray>
['202301', '202307']
Length: 2, dtype: large_string[pyarrow]

In [52]:
train_df.loc[:, ["eki_name1", "rosen_name1"]].head()

,eki_name1,rosen_name1
0,在良,三岐鉄道北勢線
1,星川,三岐鉄道北勢線
2,蓮花寺,三岐鉄道北勢線
3,寺本,名鉄常滑線
4,寺本,名鉄常滑線


In [ ]:
set(train_df.columns) - set(test_df.columns)

{'money_room'}

In [23]:
set(test_df.columns)- set(train_df.columns) 

set()

In [ ]:
# カテゴリカル変数を文字列型に変更
train_df = train_df.fillna(np.nan)
test_df = test_df.fillna(np.nan)

categorical_cols = [c for c in train_df.columns if c not in numerical_columns]

for c in categorical_cols:
    train_df[c] = train_df[c].astype(str)
    test_df[c] = test_df[c].astype(str)

In [24]:
data_definition = pd.ExcelFile(f"{ROOT_DIR}/data_definition.xlsx")

In [25]:
train_df[['addr1_1', 'addr1_2']].sample(3, random_state=2025)

,addr1_1,addr1_2
48171,11,201
16175,27,147
286956,1,106


In [ ]:
train_df[['addr1_1', 'addr1_2']].dtypes

addr1_1    object
addr1_2    object
dtype: object

In [26]:
# data_definition.sheet_names
import polars as pl

codes = pd.read_excel(f"{ROOT_DIR}/data_definition.xlsx", sheet_name=data_definition.sheet_names[3])
codes.columns = ['No.', 'addr1_1', 'addr1_2', 'prefecture_name',
       'city_town_village_name']
codes = codes[['addr1_1', 'addr1_2', 'prefecture_name',
       'city_town_village_name']]

# train_df = pd.merge(train_df, codes, on=['addr1_1', 'addr1_2'], how='inner')
# test_df = pd.merge(test_df, codes, on=['addr1_1', 'addr1_2'], how='inner')
train_df = pl.DataFrame(train_df)
test_df = pl.DataFrame(test_df)
codes = pl.DataFrame(codes)

codes = codes.with_columns(
    pl.col("addr1_2").cast(pl.Int64).cast(pl.String)
)
codes = codes.with_columns(
    pl.col("addr1_1").cast(pl.String)
)

In [27]:
train_df = train_df.join(codes, on=['addr1_1', 'addr1_2'], how='inner')
test_df = test_df.join(codes, on=['addr1_1', 'addr1_2'], how='inner')
del codes

In [28]:
train_df[['prefecture_name', 'city_town_village_name']].sample(3, seed=2025)

prefecture_name,city_town_village_name
str,str
"""静岡県""","""浜松市南区"""
"""京都府""","""京都市右京区"""
"""埼玉県""","""入間市"""


## スラッシュ区切りの列を展開

In [ ]:
slashed_columns = ["building_tag_id", "unit_tag_id", 
                   "reform_interior",  "reform_exterior","reform_wet_area",
                  "statuses"]

In [ ]:
import polars as pl

# slashed_columns: スラッシュ区切りの列名リスト
# tag_master: {タグID: タグ名} の dict を想定

def get_slashed_tags(df: pl.DataFrame) -> pl.DataFrame:
    """スラッシュ区切りの値を持つ列を個別の列に変換する（Polars版）"""

    # 行を識別するためのIDを付与
    df_idx = df.with_row_index("row_id")

    temp_dfs: list[pl.DataFrame] = []

    for i, col in enumerate(slashed_columns):
        # "a/b/c" → ["a", "b", "c"] に分割して縦持ちに
        tags = (
            df_idx
            .select(
                "row_id",
                pl.col(col).str.split("/").alias("tag")
            )
            .explode("tag")  # 1行複数タグ → 1タグ1行
        )

        # tag 列をダミー変数に変換（row_id はそのまま残る）
        dummies = tags.to_dummies(columns="tag")

        # 同じ row_id で複数行になっているので、0/1 の max でまとめる
        col_df = (
            dummies
            .group_by("row_id")
            .max()
        )

        # row_id 以外がダミー列
        dummy_cols = [c for c in col_df.columns if c != "row_id"]

        # まず 0/1 を文字列に変換（pandas の .astype("str") 相当）
        col_df = col_df.with_columns(
            [pl.col(c).cast(pl.Utf8) for c in dummy_cols]
        )

        # カラム名 "tag_123" → "元の列名 タグ名" に変換
        rename_map: dict[str, str] = {}
        for old in dummy_cols:
            # to_dummies の列名は "tag_<ID>" になっている想定
            tag_id = old.split("_", 1)[1].replace(" ", "") if "_" in old else old
            # if col == "reform_interior":
            #     tag_name = tag_reform_interior.get(tag_id, tag_id)
            # elif col == "reform_exterior":
            #     tag_name = tag_reform_exterior.get(tag_id, tag_id)
            # elif col == "reform_wet_area":
            #     tag_name = tag_reform_wet_area.get(tag_id, tag_id)
            # elif col == "statuses":
            #     tag_name = tag_statuses.get(tag_id, tag_id)
            # else:
            #     tag_name = tag_master.get(tag_id, tag_id)
            new_name = f"{col}_{tag_id}"
            rename_map[old] = new_name

        col_df = col_df.rename(rename_map)

        # 2列目以降は row_id が重複するので削る
        if i > 0:
            col_df = col_df.drop("row_id")

        temp_dfs.append(col_df)

    # 列方向に結合
    result = pl.concat(temp_dfs, how="horizontal")

    # row_id は不要なので削除（行順は row_id 順のまま）
    result = result.sort("row_id").drop("row_id")

    return result


### train_dfとtest_dfで個別に実行できますが、`pandas get_dummies` を使用しているため、testとtrainのデータフレームで列数が一致しない可能性があります。そこで、結合 -> 列を展開 -> 再度分割します。

## スラッシュ区切りの列を取得
この処理には約**1分ほど**かかります。

In [ ]:
combined_df = pl.concat([train_df, test_df], how="vertical_relaxed")

# 文字列型の列に対する前処理
str_cols = [c for c, t in zip(combined_df.columns, combined_df.dtypes) if t == pl.Utf8]

# スペースを削除
combined_df = combined_df.with_columns([
    pl.col(c).str.replace_all(" ", "").alias(c)
    for c in str_cols
])

slashed_df = get_slashed_tags(combined_df)

In [ ]:
# 新しく生成された列名を保存
tag_columns = slashed_df.columns
# 抽出した特徴量を結合
combined_df = pl.concat([combined_df, slashed_df], how="horizontal")
# スラッシュ区切りの列を削除
combined_df = combined_df.drop(slashed_columns)
# trainとtestに分割
train_df = combined_df[:len(train_df)]
test_df = combined_df[len(train_df):]

In [ ]:
del slashed_df
gc.collect()

309

In [ ]:
train_df.shape, test_df.shape

((363924, 519), (112437, 519))

## 築年を処理し、年のみを保持

In [ ]:
def parse_year(date_input):
    try:
        date_input = str(date_input)
        return date_input[:4]           
    except Exception as e:
        return str(date_input)

In [ ]:
train_df["year_built"].sample(3)

year_built
str
"""199111.0"""
"""201810.0"""
"""nan"""


In [ ]:
train_df = train_df.with_columns(
    pl.col("year_built").map_elements(parse_year)
)

test_df = test_df.with_columns(
    pl.col("year_built").map_elements(parse_year)
)


In [ ]:
train_df["year_built"].sample(3)

year_built
str
"""2018"""
"""2009"""
"""1967"""


### カテゴリカル列の中には、int, float, stringなど混在したデータ型があります. 簡潔にするために、すべて文字列に変換します

In [ ]:
categorical_cols = [c for c in train_df.columns if c not in numerical_columns]

In [ ]:
train_df = train_df.to_pandas(use_pyarrow_extension_array=True)
test_df = test_df.to_pandas(use_pyarrow_extension_array=True)

In [ ]:
train_df[categorical_cols[-1]].head()

0    0
1    0
2    0
3    0
4    0
Name: statuses_nan, dtype: large_string[pyarrow]

In [ ]:
# test_dfの"prefecture_name"と"city_town_village_name"の割合に応じて、train_dfからvalidation用データをサンプリング

# test_dfの割合を計算
test_ratios = (
    test_df.groupby(["prefecture_name", "city_town_village_name"]).size() / len(test_df)
).reset_index(name="ratio")

# train_dfに割合をマージ
train_with_ratios = train_df.merge(
    test_ratios, 
    on=["prefecture_name", "city_town_village_name"], 
    how="left"
)

# 欠損値を0で埋める（test_dfに存在しない組み合わせの場合）
train_with_ratios["ratio"] = train_with_ratios["ratio"].fillna(0)

# 各グループの割合に基づいてサンプリング
def stratified_sample(group, ratio):
    n_samples = int(len(group) * ratio)
    return group.sample(n=n_samples, random_state=42)

validation_data = train_with_ratios.groupby(["prefecture_name", "city_town_village_name"], group_keys=False).apply(
    lambda group: stratified_sample(group, group["ratio"].iloc[0])
)

# 残りを再度trainデータとする
train_data = train_df.drop(validation_data.index)

# 目的変数も分割
validation_money_room = train_money_room[validation_data.index]
train_money_room = train_money_room[train_data.index]

addr1_1  addr1_2
13       112        0.010931
         121        0.009872
11       203        0.008912
13       111        0.008769
         108        0.008698
         119        0.008165
12       204        0.008102
27       210        0.008031
13       104        0.007862
27       227        0.007675
13       123        0.007604
         103        0.007560
         201        0.007542
28       201        0.007498
13       120        0.007444
28       204        0.006670
13       109        0.006421
12       203        0.006146
13       115        0.006101
         122        0.006083
30       201        0.005834
12       207        0.005550
25       201        0.005399
13       113        0.005399
14       201        0.005292
28       202        0.005221
13       110        0.005025
27       215        0.004972
46       201        0.004954
11       201        0.004874
39       201        0.004812
28       203        0.004749
13       105        0.004651
26       109        0.0046

In [ ]:
# 前処理したデータを保存
train_df.to_parquet("../dataset/processed_train.parquet", index=False)
test_df.to_parquet("../dataset/processed_test.parquet", index=False)